In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC ### Capa Silver: Limpieza y Unión
# MAGIC Transformamos los datos y los unimos por fecha.

from pyspark.sql.functions import col, to_date, date_format

# 1. Leer las tablas Bronze
df_ventas = spark.table("ecommerce_catalog.bronze.ventas_raw")
df_pedidos = spark.table("ecommerce_catalog.bronze.pedidos_raw")

# 2. Limpieza de Ventas
df_ventas_clean = df_ventas.withColumn("fecha", to_date(col("Date"), "dd-MM-yyyy")) \
                           .drop("Date", "ingestion_date") \
                           .withColumnRenamed("Product_Category", "categoria_producto") \
                           .withColumnRenamed("Price", "precio") \
                           .withColumnRenamed("Discount", "descuento") \
                           .withColumnRenamed("Customer_Segment", "segmento_cliente") \
                           .withColumnRenamed("Marketing_Spend", "gasto_marketing") \
                           .withColumnRenamed("Units_Sold", "unidades_vendidas")

# 3. Limpieza de Pedidos
# CORRECCIÓN: Usamos try_cast para evitar que falle si encuentra texto, 
# y filtramos los valores nulos que resulten de esa conversión.
df_pedidos_clean = df_pedidos \
    .withColumn("order_id_temp", col("order_id").try_cast("bigint")) \
    .filter(col("order_id_temp").isNotNull()) \
    .drop("order_id") \
    .withColumnRenamed("order_id_temp", "order_id") \
    .withColumn("order_hour_of_day", col("order_hour_of_day").cast("int")) \
    .withColumn("order_dow", col("order_dow").cast("int")) \
    .withColumnRenamed("order_id", "id_pedido") \
    .withColumnRenamed("user_id", "id_usuario") \
    .withColumnRenamed("order_hour_of_day", "hora_pedido") \
    .withColumnRenamed("order_dow", "dia_semana_pedido")

# 4. Guardar tablas Silver
df_ventas_clean.write.mode("overwrite").saveAsTable("ecommerce_catalog.silver.ventas_clean")
df_pedidos_clean.write.mode("overwrite").saveAsTable("ecommerce_catalog.silver.pedidos_clean")

print("Capa Silver completada con éxito. Tablas creadas: ventas_clean, pedidos_clean")

Capa Silver completada con éxito. Tablas creadas: ventas_clean, pedidos_clean
